# 03: データセットの確認と再生

配布された**この回のフォルダ一式**を使います。ipynbの隣のPython・JSONファイルを読み込みます。
ライブラリと公開モデル・データは実行時に取得します。教材リポジトリのcloneや、隠れたコードの展開は行いません。

1. この回のフォルダをDriveに置き、このipynbをColabで開きます。
2. ランタイムを **CPU（ハードウェアアクセラレータなし）** にし、最初のセルにフォルダのパスを指定します。
3. 説明→準備→設定→実行→結果確認→保存の順で進めます。

00・01はLift、02以降はLIBEROを扱います。短い接続確認と、成功率を比較する評価は区別します。
T4で短い動作確認を実施しています。長時間学習や最終評価は各実習で行います。

今回使うファイル：
- `check_environment.py`
- `common.py`
- `dataset.json`
- `dataset_tools.py`
- `prepare.py`
- `requirements-py312.txt`
- `requirements-py313.txt`

In [ ]:
# @title この回の配布ファイルを準備する
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

USE_DRIVE = True  # @param {type:"boolean"}
LESSON_DIR = "/content/drive/MyDrive/vla_Workshop/03_dataset"  # @param {type:"string"}
REQUIRED_FILES = [
    "check_environment.py",
    "common.py",
    "dataset.json",
    "dataset_tools.py",
    "prepare.py",
    "requirements-py312.txt",
    "requirements-py313.txt",
]

if USE_DRIVE:
    drive.mount("/content/drive")
    source = Path(LESSON_DIR)
else:
    # ipynb以外の今回の配布ファイルを、まとめて選んでアップロードします。
    os.chdir("/content")
    files.upload()
    source = Path("/content")
missing = [name for name in REQUIRED_FILES if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"配布ファイルが不足: {missing}。LESSON_DIRを確認してください。")
PROJECT = Path("/content/lesson_03")
PROJECT.mkdir(exist_ok=True)
for name in REQUIRED_FILES:
    shutil.copy2(source / name, PROJECT / name)
os.chdir(PROJECT)
print("この回の作業場所:", PROJECT)

## ライブラリの準備

この回だけの専用Pythonを作ります。初回はダウンロードに時間がかかります。教材のPythonパッケージはインストールしません。

In [ ]:
# @title ライブラリを準備する
LESSON_ENV = {
    **os.environ,
    "MPLBACKEND": "Agg",
    "PYTHONPATH": str(PROJECT),
    "PYTHONNOUSERSITE": "1",
}
command = [sys.executable, str(PROJECT / "prepare.py"), "--project", str(PROJECT)]
command.append("--cpu")
subprocess.run(command, env=LESSON_ENV, check=True)
PYTHON = PROJECT / ".venv/bin/python"


def run_python(program):
    """セルに書いたPythonを、この回の専用Pythonで実行する。"""
    last_result = None
    with subprocess.Popen(
        [str(PYTHON), "-u", "-c", program],
        cwd=PROJECT,
        env=LESSON_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            if line.startswith("Results: "):
                last_result = Path(line.removeprefix("Results: ").strip())
        if process.wait():
            raise RuntimeError("実行に失敗しました。直前のエラーを確認してください。")
    return last_result

## データと分割を確認する

In [ ]:
config = json.loads((PROJECT / "dataset.json").read_text())
# train_episodesとvalidation_episodesは重複させません。
print(json.dumps(config, indent=2, ensure_ascii=False))
(PROJECT / "active_dataset.json").write_text(json.dumps(config))

## データを読み込み、検証して再生する

In [ ]:
result = run_python(r"""
from common import read_json
from dataset_tools import inspect_dataset

inspect_dataset(read_json("active_dataset.json"))
""")

## 観測・行動・動画の対応を見る

In [ ]:
from IPython.display import Video, display

display(Video(str(result / "replay.mp4"), embed=True))
print((result / "dataset.json").read_text())

## 確認課題

- 入力・出力と、変更した条件を説明できますか。
- 結果のファイルと動画を開き、失敗や不足を確認しましたか。
- 次の回で使う保存先を記録しましたか。

05では採否を確認し、06では学習・検証のエピソードを分けます。07では検証結果で候補を決めてから最終評価を行います。

## 結果をDriveに保存する

動画・ログ・モデルなど、この回のoutputsを保存します。次の回でデータ・モデルを指定できるよう、フォルダのまま保存します。保存後にランタイムを接続解除して削除してください。

In [ ]:
# @title 結果を保存する
from datetime import UTC, datetime
from uuid import uuid4

RESULTS_DIR = "/content/drive/MyDrive/vla_Workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
name = "03_dataset_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
saved = destination / name
shutil.copytree(PROJECT / "outputs", saved)
print("保存先:", saved)
for info in sorted(saved.glob("*/dataset/meta/info.json")):
    print("06のDATASET_ROOT:", info.parents[1])
for model in sorted(saved.glob("*/pretrained_model/config.json")):
    print("07のpolicy.path:", model.parent)